In [5]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [1]:
# === 샘플 문서 준비 ===
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고,
사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# === ChatPromptTemplate 생성 ===
# ("역할", "내용") 튜플 리스트로 메시지 구조를 정의합니다.
# {instruction}, {document}, {question}이 변수 자리표시자입니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 참고하여 질문에 답하세요. {instruction}"),
    ("human", "[document]\n{document}\n\n[질문]\n{question}")
])

# === 변수 목록 확인 ===
print(f"필요한 변수: {prompt.input_variables}")

필요한 변수: ['document', 'instruction', 'question']


In [ ]:
# === invoke(): 변수를 주입하여 메시지 리스트 생성 ===
result = prompt.invoke({
    "instruction": "문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요.",
    "document": document,
    "question": "어댑터즈에서 사용하는 교수법은 무엇인가요?"
})

# === 생성된 메시지 확인 ===
for msg in result.messages: # type: ignore
    print(f"[{msg.type}] {msg.content[:60]}...")

[system] 다음 문서를 참고하여 질문에 답하세요. 문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'...
[human] [document]
Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑...


In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)
parser = StrOutputParser()

# === 파이프(|) 연산자로 체인 구성 ===
# ChatPromptTemplate → Chat Model → 출력 파서를 순서대로 연결합니다.
chain = prompt | model | parser

# === 체인 실행 ===
result = chain.invoke({
    "instruction": "문서에 없는 내용은 '제공된 문서에서 해당 정보를 찾을 수 없습니다'라고 답하세요.",
    "document": document,
    "question": "어댑터즈에서 사용하는 교수법은 무엇인가요?"
})
print(result)

어댑터즈에서 사용하는 교수법은 '5단 분석법'입니다.


In [8]:
# === 같은 템플릿, 다른 질문 ===
# 프롬프트 구조는 그대로 두고 질문만 바꿔서 반복 사용합니다.
questions = [
    "어댑터즈에서 사용하는 교수법은 무엇인가요?",
    "어댑터즈는 어떤 분야의 교재를 제공하나요?",
    "어댑터즈의 운영 회사는 어디인가요?",
]

for q in questions:
    result = chain.invoke({
        "instruction": "한 문장으로 답하세요.",
        "document": document,
        "question": q
    })
    print(f"Q: {q}")
    print(f"A: {result}")
    print()

Q: 어댑터즈에서 사용하는 교수법은 무엇인가요?
A: 어댑터즈에서 사용하는 교수법은 5단 분석법입니다.

Q: 어댑터즈는 어떤 분야의 교재를 제공하나요?
A: 어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다.

Q: 어댑터즈의 운영 회사는 어디인가요?
A: 어댑터즈의 운영 회사는 스타트업코드입니다.

